# Resume an Existing Atlas

This tutorial shows how to reopen an existing `.sasql` Atlas, inspect the
persisted analysis state, and continue a workflow in a later Python session
without importing the source data again.

Use this tutorial when you have already completed part of an analysis and want
to determine which results are available and which step should be run next.

By the end of this tutorial, you will be able to:

- safely reconnect to an existing Atlas;
- inspect imported data, metadata, and analysis results;
- review the current read index;
- identify the appropriate point from which to resume the workflow;
- continue analysis without unnecessarily recomputing existing results.

## Before You Begin

Resuming an Atlas restores information that has been written to the `.sasql`
database, including expression data, metadata, embeddings, cluster labels, and
other stored analysis results.

Objects that existed only in the previous Python session are not restored
automatically. These may include:

- Python variables other than data stored in the Atlas;
- active data iterators and minibatch generators;
- fitted scikit-learn or PyTorch models that were not saved separately;
- temporary arrays, data frames, and plotting objects.

Save external models and other Python objects separately when they are required
in a later session.


## 1. Verify the Atlas Path

Check that the expected `.sasql` file exists before constructing the `Atlas`
object:


In [5]:
import os
from pathlib import Path
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_resume.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")


Checking the path first helps prevent a misspelled path from being mistaken for
an existing analysis.


## 2. Open the Atlas

Create a new `Atlas` object using the same database path:


In [ ]:
atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)

scAtlasPy connects to the existing database and makes its stored data and
analysis results available in the current Python session.

`db_memory_limit` configures the DuckDB connection created for this session. It
does not modify the expression data or analysis results already stored in the
database.


## 3. Inspect the Atlas Summary

Begin with a general summary:


In [7]:
atlas.describe()

'file_name       : /home/hanxu/scAtlaspy-code-analysis/tmp/tutorials/basic_pbmc3k/pbmc3k_basic_resume.sasql\ndb_memory_limit : 8GB\ntables          : 17\ntable names     : X_HyS_data, X_HyS_data_filtered, X_HyS_indptr, X_HyS_indptr_filtered, atlas_read_index_meta, kmeans_centers, manual_cluster_annotation, obs, obs_cluster, obsm_X_pca, obsm_X_umap, rank_genes_groups, uns_pca_stats, uns_umap_eval, uns_umap_params, var, varm_PCs\nn_cells         : 2,700\nn_genes         : 32,738'

Preview the cell and gene metadata:


In [ ]:
atlas.head("obs", n=5)

table   : obs
columns : atlas_cell_id, atlas_cell_name, filter_cells, cell_total_counts, n_genes_by_counts, total_counts_mt, pct_counts_mt, total_counts_ribo, pct_counts_ribo, scale_factor, filter_cell_id, kmeans, cell_type_manual
rows    : first 5
   atlas_cell_id   atlas_cell_name  filter_cells  cell_total_counts  n_genes_by_counts  total_counts_mt  pct_counts_mt  total_counts_ribo  pct_counts_ribo  scale_factor  filter_cell_id  kmeans cell_type_manual
0              0  GAACCTGAACGTGT-1          True             4756.0               1491            152.0       3.195963             1826.0        38.393608      2.102607               0       7            CD4 T
1              1  AATCTCTGCTTTAC-1          True             1710.0                668             31.0       1.812865              516.0        30.175438      5.847953               1       6               NK
2              2  GGACCTCTGTAAGA-1          True             2399.0                900             78.0       3.251355   

Check that important metadata fields, such as sample, donor, batch, condition,
cluster, or cell-type annotations, are present.

You can inspect the schemas of the metadata tables with:


In [9]:
atlas.query("PRAGMA table_info(obs)")
atlas.query("PRAGMA table_info(var)")


,cid,name,type,notnull,dflt_value,pk
0,0,atlas_gene_id,USMALLINT,True,None,True
1,1,atlas_gene_name,VARCHAR,False,None,False
2,2,gene_ids,VARCHAR,False,None,False
3,3,filter_genes,BOOLEAN,False,CAST('f' AS BOOLEAN),False
4,4,mt,BOOLEAN,False,None,False
5,5,ribo,BOOLEAN,False,None,False
6,6,gene_total_counts,FLOAT,False,None,False
7,7,n_cells_by_counts,INTEGER,False,None,False
8,8,highly_variable_genes,BOOLEAN,False,None,False
9,9,highly_variable_rank,FLOAT,False,None,False


## 4. Review the Persisted Workflow State

The following artifacts can help identify which analysis steps have already
been completed.

| Persisted artifact | What it usually indicates |
|---|---|
| `obs` and `var` tables | Data have been imported. |
| `filter_cells` column in `obs` | A cell-filtering result has been stored. |
| `filter_genes` column in `var` | A gene-filtering result has been stored. |
| `highly_variable_genes` column in `var` | Highly variable genes have been selected. |
| `atlas_read_index_meta` table | A read index has been constructed. |
| `obsm_X_pca` and `varm_PCs` tables | PCA coordinates and loadings have been stored. |
| `kmeans` column in `obs` | KMeans cluster labels have been stored. |
| `obsm_X_umap` table | UMAP coordinates have been stored. |

```{important}
The presence of a table or metadata column shows that a result has been stored,
but it does not by itself guarantee that the result matches the current
filtering conditions, read index, or expression representation.

Before reusing an existing result, confirm that it was calculated from the cell
set, gene set, and expression field intended for the current analysis.
```


## 5. Inspect the Current Read Index

Many atlas-scale algorithms and streaming workflows operate through a read
index. The read index determines:

- which cells are included;
- which genes are included;
- whether highly variable genes are used;
- which expression field is read.

If `atlas_read_index_meta` is present, inspect its contents:


In [10]:
atlas.query("""
    SELECT key, value
    FROM atlas_read_index_meta
    ORDER BY key
""")


,key,value
0,cell_condition,filter_cells
1,gene_condition,filter_genes
2,use_data,data_scale
3,use_hvg,True


Confirm that its cell selection, gene selection, and expression field match the
analysis you want to continue.

For example, a preprocessing workflow may use:


In [11]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_log1p",
)


build_read_index:   0%|          | 0/2286884 [00:00<?, ?rows/s]

In this example:

- only cells passing `filter_cells` are included;
- only genes passing `filter_genes` are included;
- the selection is further restricted to highly variable genes;
- downstream streaming methods read `data_log1p`.

```{warning}
Do not rebuild the read index unless its current configuration is missing or
does not match the intended analysis.

Changing the read index does not automatically recompute existing PCA,
clustering, UMAP, or model results. Results calculated with an earlier cell
selection, gene selection, or expression field may no longer be consistent with
the new read index and should be recomputed when necessary.
```


## 6. Choose Where to Resume

Use the persisted state to identify the next analysis step.

| Current Atlas state | Typical next action |
|---|---|
| Expression data have been imported, but preprocessing is incomplete | Continue with quality control, filtering, normalization, and feature selection. |
| Preprocessing is complete, but no suitable read index exists | Construct the read index for the intended analysis. |
| The read index is ready, but PCA is missing | Run PCA. |
| PCA is available, but clustering is missing | Run the selected clustering method. |
| PCA is available, but UMAP is missing | Calculate UMAP coordinates. |
| Analysis results are complete | Inspect visualizations, query stored results, annotate cells, or export data. |

Run only the steps that are missing or that need to be recalculated.

For example, if preprocessing and read-index construction are complete but PCA
has not yet been calculated:


In [12]:
sap.tl.pca(
    atlas,
    n_components=50,
    fit_batches=1000,
)


PCA:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

If PCA is already available but KMeans clustering is missing:


In [ ]:
sap.tl.kmeans(
    atlas,
    n_clusters=10,
    fit_batches=1000,
)


If the required upstream results are already present, continue directly to the
corresponding downstream step rather than rerunning the entire workflow.

```{note}
Whether an existing result should be reused depends on more than its presence
in the database. Recompute downstream results when their upstream cell
selection, gene selection, expression representation, or parameters have
changed.
```


## 7. Close the Connection

Close the database connection when the current session is complete:


In [ ]:
atlas.close()


Closing the connection does not delete the `.sasql` file or its stored results.
The same Atlas can be reopened in another Python session using its database
path.

## Next Steps

- See {doc}`visualize-analysis-results` to inspect quality-control,
  dimensionality-reduction, clustering, and marker-analysis results.
- See {doc}`query-atlas-with-sql` to explore metadata and analysis results
  directly with SQL.
- Return to the {doc}`../basic/index` if quality control or preprocessing has
  not yet been completed.
